#### Import

In [1]:
import numpy as np; import pandas as pd; import matplotlib.pyplot as mp; import seaborn as sea; import datetime as dt; import random as r; import math as m; import statistics as st; import mysql.connector as ms; import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy connectable*"); warnings.filterwarnings("ignore", message=".*is_categorical_dtype is deprecated.*"); warnings.filterwarnings("ignore", message="invalid escape sequence.*", category=SyntaxWarning)


#### Data

In [2]:
dbd="titanic"


#### Columns

In [4]:
db = ms.connect(host = "localhost", 
                username = "root", 
                password = "67777777", 
                database = dbd) 
cur = db.cursor()
cur.execute("SHOW TABLES");tables=[t[0] for t in cur.fetchall()];dfs=[];import pandas as pd
for i,table in enumerate(tables,1):
 df=pd.read_sql(f"SELECT * FROM {table} LIMIT 10",con=db)
 for col in df.columns:
  if 'date'in col.lower()or'time'in col.lower():df[col]=pd.to_datetime(df[col],errors='coerce')
 temp=pd.DataFrame({'Table':table,'Column':df.columns,'Type':[str(df[c].dtype)for c in df.columns]});temp.index=[i]*len(temp);temp.index.name='index';dfs+=[temp,pd.DataFrame([['','','']],columns=temp.columns,index=[""])]
cn_final=pd.concat(dfs);print(f"\n{'':<8}{'Table':<20}{'Column':30}{'Type'}\n")
for i,table in enumerate(tables,1):
 for _,r in cn_final[cn_final['Table']==table].iterrows():print(f"{i:<5}{r['Table']:<18}{r['Column']:<35}{r['Type']}")
 print()
print("All tables in the database:\n"); 
for table in tables: print(f"{table}")


        Table               Column                        Type

1    survive           PassengerId                        int64
1    survive           Survived                           int64

2    titanic           passengerid                        int64
2    titanic           pclass                             int64
2    titanic           name                               str
2    titanic           sex                                str
2    titanic           age                                int64
2    titanic           sibsp                              int64
2    titanic           parch                              int64
2    titanic           ticket                             int64
2    titanic           fare                               float64
2    titanic           cabin                              int64
2    titanic           embarked                           str
2    titanic           name_isnull                        int64
2    titanic           age_isnull         

#### Table

In [5]:
table="titanic"
try:display(pd.read_sql(f"select * from {table}", db).pipe(lambda df: df.rename(columns=lambda c: f"{c} (c{df.columns.get_loc(c)+1})").head().style.hide(axis="index").set_properties(**{'text-align':'left'}).set_table_styles([{'selector':'th','props':[('text-align','center'),('border','1.5px solid white')]},{'selector':'td','props':[('border','1px solid white')]}])))
except:pass

passengerid (c1),pclass (c2),name (c3),sex (c4),age (c5),sibsp (c6),parch (c7),ticket (c8),fare (c9),cabin (c10),embarked (c11),name_isnull (c12),age_isnull (c13),fare_isnull (c14),cabin_isnull (c15)
988,1,Tyrell William,F,76,1,0,19877,78.850000,1,S,1,1,1,1
1071,1,Alexander Taylor,F,64,0,2,17756,83.160000,1,C,1,1,1,1
1197,1,Edward Gifford,F,64,1,1,112901,26.550000,1,S,1,1,1,1
1006,1,Isidor,F,63,1,0,17483,221.780000,1,S,1,1,1,1
940,1,William Robert,F,60,0,0,11813,76.290000,1,C,1,1,1,1


#### 1. What is the total number of passengers and how many survived vs did not survive

In [6]:
query=""" 
select count(ti.passengerid) as passengerid,
sum(
case
when su.survived=1 then 1
else 0
end
) as survived,
sum(
case
when su.survived=0 then 1
else 0
end
) as not_survived
from titanic ti join survive su using(passengerid)
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


passengerid (c1),survived (c2),not_survived (c3)
418,152,266


#### 2. What is the gender distribution of passengers

In [7]:
query=""" 
select sex , 
count(sex) as genders
from titanic 
group by sex
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


sex (c1),genders (c2)
F,152
M,266


#### 3. What is the average, median, and range of passenger ages

In [8]:
query=""" 
select * from titanic
""" 
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
median = df.age.median()

query=f""" 
select round(avg(age),2) as age_average,
round({median},0) as age_median,
concat(min(age),"-",max(age)) as age_range
from titanic
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


age_average (c1),age_median (c2),age_range (c3)
30.11,28,1-76


#### 4. How many passengers belong to each class (Pclass)

In [9]:
query=""" 
select pclass , count(pclass) as count_of_pclass
from titanic
group by pclass
order by pclass asc
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


pclass (c1),count_of_pclass (c2)
1,107
2,93
3,218


#### 5. What is the survival rate by gender

In [10]:
query=""" 
select ti.sex ,
concat(
round(100*sum(
case
when su.survived=1 then 1
else 0
end 
)/count(ti.sex)
,0)
,"%")
as survival_rate
from titanic ti
join survive su using(passengerid)
group by ti.sex
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


sex (c1),survival_rate (c2)
F,100%
M,0%


#### 6. What is the survival rate by passenger class

In [11]:
query="""
select ti.pclass as pclass,
concat(
round(100*sum(
case
when su.survived=1 then 1
else 0
end 
)/count(ti.pclass)
,1)
,"%")
as survival_rate
from titanic ti
join survive su using(passengerid)
group by ti.pclass 
order by pclass asc

"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


pclass (c1),survival_rate (c2)
1,46.7%
2,32.3%
3,33.0%


#### 7. What is the average fare paid by each class

In [12]:
query=""" 
select pclass ,
round(avg(fare),1) as dd
from titanic
group by pclass
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))

pclass (c1),dd (c2)
1,94.300000
2,22.200000
3,12.500000


#### 8. Which embarkation port (Embarked) has the highest number of passengers

In [13]:
query=""" 
select embarked ,
count(*) as count_of_embarked
from titanic
group by embarked
order by count_of_embarked desc
limit 1
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


embarked (c1),count_of_embarked (c2)
S,270


#### 9. Analyze survival rate by both gender and class (e.g., female in 1st class vs male in 3rd class).

In [14]:
query=""" 
SELECT 
    pclass,
    ROUND(100.0 * AVG(CASE WHEN sex = 'F' THEN su.survived END), 1) AS female_rate,
    ROUND(100.0 * AVG(CASE WHEN sex = 'M' THEN su.survived END), 1) AS male_rate
FROM titanic ti
JOIN survive su USING(passengerid)
GROUP BY pclass;
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


pclass (c1),female_rate (c2),male_rate (c3)
1,100.0,0.0
2,100.0,0.0
3,100.0,0.0


#### 10. Identify if there is a relationship between fare and survival (use grouping or visualization logic).

In [15]:

query=""" 
SELECT 
    case
    when fare_group=1 then "High"
    when fare_group=2 then "Medium"
    else "Low"
    end as fare_group ,
    ROUND(100.0 * AVG(survived),1) AS survival_rate
FROM (
    SELECT 
        su.survived as survived,
        NTILE(3) OVER (ORDER BY ti.fare desc) AS fare_group
    FROM titanic ti
    JOIN survive su USING(passengerid)
) t
GROUP BY fare_group;
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))
print("Yes there is a relationship")

fare_group (c1),survival_rate (c2)
High,49.3
Medium,32.4
Low,27.3


Yes there is a relationship


#### 11. Segment passengers into age groups and compare survival rates across groups.

In [16]:
query=""" 
with a as (

select age,passengerid,
case
when age<12 then "Child"
when age<18 then "Teen"
when age<40 then "Adult"
else "Senior"
end as passenger_group,
case
when age<12 then 1
when age<18 then 2
when age<40 then 3
else 4
end as passenger_group_order

from titanic
)

select pg.passenger_group,
round(100.0*
sum(case
when su.survived=1 then 1
else 0
end )/count(*)
,2)
as survival_rate
from a pg 
left join survive su using(passengerid)
group by pg.passenger_group,pg.passenger_group_order
order by pg.passenger_group_order

"""
data="date" 
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


passenger_group (c1),survival_rate (c2)
Child,44.44
Teen,38.89
Adult,35.67
Senior,36.59


#### 12. Identify outliers in fare and age - how do they affect your analysis

In [17]:
query=""" 
with ranked_fare as (
select fare,
ntile(4) over(order by fare) as quartile
from titanic
)
,
quartiles_fare as (
select 
min(case when quartile=1 then fare end) as q1,
min(case when quartile=3 then fare end) as q3
from ranked_fare
)

,ranked_age as (
select age,
ntile(4) over(order by age) as quartile
from titanic
)
,
quartiles_age as (
select 
min(case when quartile=1 then age end) as q1,
min(case when quartile=3 then age end) as q3
from ranked_age
)

select t.* 
from titanic t 
cross join quartiles_fare fq
cross join quartiles_age aq
where t.fare < (fq.q1 - 1.5 * (fq.q3 - fq.q1))
or    t.fare > (fq.q3 + 1.5 * (fq.q3 - fq.q1))
or    t.age < (aq.q1 - 1.5 * (aq.q3 - aq.q1))
or    t.age > (aq.q3 + 1.5 * (aq.q3 - aq.q1))


"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align':'left'}).set_table_styles([{'selector':'th','props':[('text-align','center')]},{'selector':'th','props':[('border','1.5px solid white')]},{'selector':'td','props':[('border','1px solid white')]}]))


passengerid (c1),pclass (c2),name (c3),sex (c4),age (c5),sibsp (c6),parch (c7),ticket (c8),fare (c9),cabin (c10),embarked (c11),name_isnull (c12),age_isnull (c13),fare_isnull (c14),cabin_isnull (c15)
988,1,Tyrell William,F,76,1,0,19877,78.850000,1,S,1,1,1,1
1071,1,Alexander Taylor,F,64,0,2,17756,83.160000,1,C,1,1,1,1
1006,1,Isidor,F,63,1,0,17483,221.780000,1,S,1,1,1,1
940,1,William Robert,F,60,0,0,11813,76.290000,1,C,1,1,1,1
961,1,Mark,F,60,1,4,19950,263.000000,1,S,1,1,1,1


#### 13. Write SQL-style logic to calculate survival rate by multiple dimensions (e.g., gender + class).

In [18]:
query=""" 
SELECT 
    sex,
    pclass,
    COUNT(*) AS total_passengers,
    
    SUM(CASE WHEN survived = 1 THEN 1 ELSE 0 END) AS survivors,
    
    ROUND(
        SUM(CASE WHEN survived = 1 THEN 1 ELSE 0 END) * 1.0 
        / COUNT(*), 
        2
    ) AS survival_rate

FROM titanic left join survive using(passengerid)
GROUP BY sex, pclass
ORDER BY sex, pclass;
"""
cur.execute(query)
data=cur.fetchall()
columns = [col[0] for col in cur.description]
df = pd.DataFrame(data, columns=columns)
df = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(df.head().style.hide(axis="index").set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},{'selector': 'th', 'props': [('border', '1.5px solid white')]},{'selector': 'td', 'props': [('border', '1px solid white')]}]))


sex (c1),pclass (c2),total_passengers (c3),survivors (c4),survival_rate (c5)
F,1,50,50,1.00
F,2,30,30,1.00
F,3,72,72,1.00
M,1,57,0,0.00
M,2,63,0,0.00
